Inicializacion:

In [18]:
import numpy as np
import pandas as pd
import random as random
import copy

class Capa:
    w : np.ndarray
    y: np.ndarray
    delta: np.ndarray

    def __init__(self, w_i, y_i, delta_i):
        self.w = w_i
        self.y = y_i
        self.delta = delta_i

def sigm(x):
    return (2/(1+np.exp(-x))) - 1

entrada_usuario = [2,1]

tabla = pd.read_csv('../../Data/gtp-1/XOR_trn.csv', header=None).to_numpy()
x0 = np.ones(len(tabla))*-1
entradas = np.c_[x0,tabla[:,:-1]]
yd =  tabla[:,-1]

w = np.random.rand(entrada_usuario[0], len(entradas[0])) - 0.5
y_init = np.zeros(entrada_usuario[0])
delta = np.zeros(entrada_usuario[0])
cap = Capa(w,y_init,delta)
vect_capas = [copy.deepcopy(cap)]

for i in range(1,len(entrada_usuario)):
    w = np.random.rand(entrada_usuario[i], entrada_usuario[i-1]+1) - 0.5
    y_init = np.zeros(entrada_usuario[i])
    delta = np.zeros(entrada_usuario[i])
    cap = Capa(w,y_init,delta)
    vect_capas.append(copy.deepcopy(cap))


Entrenamiento

In [19]:
epoca = 1
epocas_max = 100 
tasa = 0.01 
tasa_aciertos = 0
# n -> ejemplo actual
# i -> la capa
# j -> la neurona
while epoca < epocas_max and tasa_aciertos<1: 
    for n in range(len(entradas)):

        #paso hacia adelante
        for i in range(len(vect_capas)):
            for j in range(len(vect_capas[i].y)):
                if i==0:
                    z = np.dot(entradas[n,:],vect_capas[i].w[j,:])
                else: 
                    ent = np.r_[-1,vect_capas[i-1].y]
                    z = np.dot(ent,vect_capas[i].w[j,:])
                vect_capas[i].y[j] = sigm(z)

        #propagacion hacia atras
        for i in range(len(vect_capas)-1,-1,-1):
            for j in range(len(vect_capas[i].y)):
                if i==len(vect_capas)-1:
                    vect_capas[i].delta[j] = (1/2) * (yd[n] - vect_capas[i].y[j]) * (1 + vect_capas[i].y[j]) * (1 - vect_capas[i].y[j])
                else:
                    vect_capas[i].delta[j] = (1/2) * np.dot(vect_capas[i+1].delta, vect_capas[i+1].w[:,j+1])  * (1 + vect_capas[i].y[j]) * (1 - vect_capas[i].y[j])

        #actualizar los pesos
        for i in range(len(vect_capas)):
            for j in range(len(vect_capas[i].y)):
                for m in range(len(vect_capas[i].w[j])):
                    if i==0:
                        vect_capas[i].w[j,m] += tasa*vect_capas[i].delta[j]*entradas[n,m]
                    else:
                        ent = np.r_[-1, vect_capas[i-1].y]
                        vect_capas[i].w[j,m] += tasa*vect_capas[i].delta[j]*ent[m]

    #validar
    aciertos = 0
    for n in range(len(entradas)):
        for i in range(len(vect_capas)):
            for j in range(len(vect_capas[i].y)):
                if i==0:
                    z = np.dot(entradas[n,:],vect_capas[i].w[j,:])
                else: 
                    ent = np.r_[-1,vect_capas[i-1].y]
                    z = np.dot(ent,vect_capas[i].w[j,:])
                vect_capas[i].y[j] = sigm(z)
        if (vect_capas[-1].y[-1] > 0.75 and yd[n] > 0) or (vect_capas[-1].y[-1] < -0.75 and yd[n] < 0) :
            aciertos +=1
    tasa_aciertos = aciertos / len(entradas)
    print(tasa_aciertos) 

    epoca += 1

0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.519
0.986
1.0


Test

In [20]:
tabla_tst = pd.read_csv('../../Data/gtp-1/XOR_tst.csv', header=None).to_numpy()

x0 = -np.ones(len(tabla_tst))
entradas = np.c_[x0, tabla_tst[:,:-1]] # indice -1 := ultima columna ( Acceso a indices con : es [) )
yd =  tabla_tst[:, -1]

acierto = 0
for n in range(len(entradas)):
    for capa in range(len(vect_capas)):
        for neuron in range(len(vect_capas[capa].y)):
            if capa==0:
                z = np.dot(entradas[n,:],vect_capas[capa].w[neuron,:])
            else:
                ent = np.r_[-1,vect_capas[capa-1].y]
                z = np.dot(ent,vect_capas[capa].w[neuron,:])

            vect_capas[capa].y[neuron] = sigm(z)

    salida_red = vect_capas[-1].y[0]
    if ((salida_red >= 0.75 and yd[n] == 1) or (salida_red <= -0.75 and yd[n] == -1)):
        acierto += 1

tasa_acierto = acierto/len(entradas)
print(f"Test: Tasa de acierto: {tasa_acierto * 100: .2f}%")

Test: Tasa de acierto:  100.00%
